In [ ]:
import sys
sys.path.append('./')
sys.path.append('/Users/ASUS/Documents/GitHub/retrospective_t2e_analysis/auton-survival/')

from gls import datasets

import pandas as pd
import numpy as np

def load_peace_trial_data(outcome, intervention='TX'):

  feature_list = {
    'prerand.sas7bdat' : ['AGE', 'DIABP', 'EGFR', 'GENDER', 'PHDIZZ', 'PHSYNC', 'SER_CHOL', 'SER_POT', 'SYSBP'],
    'rand.sas7bdat' : ['ANARRC', 'ANTICO', 'ASANT', 'BEBLOC', 'CALCBL', 'COUGH', 'CIGARE', 'CMOTH', 'DIABTC', 'DICONL',
                      'DIGITS', 'DIZZIN', 'HEADAC', 'HIANGINA', 'HIANGIOG', 'HICCABG', 'HICLAUDI', 'HIDIABET', 'HIHYPERT', 
                      'HIMI', 'HIPTCA', 'HISTROKE', 'HITIA', 'HPREP', 'INSLIN', 'LIPLOW', 'LVEDOC', 'POSPDI', 
                      'SKINRA', 'SYNCOP', 'TX', 'WT_KG', 'HT_CM', 'LVEEJF', 'MOTHER', 'NSYANG', 'ORAGEN',
                      'OTDIUR', 'POSPDI', 'QUALAB', 'QUANEF', 'TX']
  }

  if outcome is 'PRIMARY': outcomedt = 'PRIMRYDT'
  else: outcomedt = outcome+'DT'

  outcomes, _ = datasets._load_generic_biolincc_dataset('outcomes.sas7bdat', 
                                                         outcomedt, outcome,
                                                         features=feature_list, 
                                                         id_col='NEW_ID',
                                                         location='datasets/PEACE/Data/')
                                                                  
  outcomes2, features = datasets._load_generic_biolincc_dataset('outcomes.sas7bdat', 
                                                                'DAYSSINCERAND', outcome,
                                                                features=feature_list, 
                                                                id_col='NEW_ID',
                                                                location='datasets/PEACE/Data/')
  # When the outcome occurs we use the time to the outcome. 
  # Incase the outcome doesn't occur, we consider the time since randomization
  # # to last followup as the censoring time.
  outcomes.loc[outcomes.event == 0.0, 'time'] = outcomes2.loc[outcomes.event == 0.0,'time']

  # Convert Censoring Indicator to Binary
  # ie. 1 = Event, 0 = Censored
  outcomes.event = outcomes.event == 1.0 

  # intervention = features[intervention]

  # intervention[intervention == 1.0] = 'Trandolapril'
  # intervention[intervention == 0.0] = 'Placebo'

  NaNindex = pd.isna(outcomes.time)
  outcomes = outcomes.loc[~NaNindex]
  features = features.loc[~NaNindex]

  # cat_feats = ['GENDER', 'PHDIZZ', 'PHSYNC', 'ANARRC', 'ANTICO', 'ASANT', 'BEBLOC', 'CALCBL', 'CMOTH', 'COUGH',
  #              'DIABTC', 'DICONL', 'DIGITS', 'DIZZI', 'HEADAC', 'HIANGINA', 'HIANGIOG', 'HICCABG', 'HICLAUDI', 'HIDIABET', 
  #              'HIHYPERT', 'HIMI', 'HIPTCA', 'HPREP', 'INSLIN', 'LIPLOW', 'LVEDOC', 'MOTHER', 'NSYANG', 'ORAGEN', 
  #              'OTDIUR', 'QUALAB', 'QUANEF', 'POSPDI', 'SKINRA', 'SYNCOPE', 'HISTROKE', 'HITIA']

  # num_feats = ['AGE', 'DIABP', 'EGFR', 'SER_CHOL', 'SER_POT', 'SYSBP', 'CIGARE', 'SDIABP', 'LVEEJF', 'HT_CM', 'WT_KG']

  features = features.rename(columns={"AGE": "Age",
                                      "DIABP": "Seated_Diastolic_Blood_Pressure", 
                                      "PHDIZZ": "History_Dizziness",
                                      "EGFR": "Estimated_Glomerular_Filtration_Rate",
                                      "GENDER": "Sex",
                                      "PHSYNC": "History_Syncope",
                                      "SER_POT": "Serum_Potassium",
                                      "SER_CHOL": "Cholesterol", 
                                      "SYSBP": "Seated_Systolic_Blood_Pressure", 
                                      "ASANT": "Use_Aspirin/Antiplatelet_Therapy", 
                                      "CIGARE": "Baseline_Smoke_status", 
                                      "ANARRC": "Use_Anti_Arrhythmic", 
                                      "ANTICO": "Use_Anticoagulants", 
                                      "BEBLOC": "Beta_Blocker", 
                                      "CALCBL": "Calcium_Blocker", 
                                      "CMOTH": "Use_Other_Cardiac_Medication", 
                                      "COUGH": "Cough",
                                      "DIABTC": "Known_Diabetic",
                                      "HIDIABET": "History_Diabetes", 
                                      "DICONL": "Use_Diet_Control", 
                                      "DIGITS": "Use_Digitalis", 
                                      "DIZZI": "Dizziness",
                                      "HEADAC": "Headache",
                                      "HIANGINA": "History_Angina",
                                      "HIANGIOG": "Angiographic_Coronary_Disease",
                                      'HICCABG': "History_Coronary_Artery_Bypass_Graft", 
                                      'HICLAUDI': "History_Intermittent_Claudication", 
                                      'HIHYPERT': "History_Hypertension", 
                                      'HIMI': "History_Myocardial_Infarction", 
                                      'HIPTCA': "History_Percutaneous_Transluminal_Coronary_Angioplasty", 
                                      'HISTROKE': "History_Stroke", 
                                      'HITIA': "History_Transient_Ischemic_Attack", 
                                      'HPREP': "Use_Hormone_Replacement_Therapy", 
                                      'INSLIN': "Use_Insulin", 
                                      'LIPLOW': "Use_Lipid_Lowering_Therapy",
                                      "LVEDOC": "Left_Ventricular_Evaluation_Documented_By",
                                      "LVEEJF": "Left_Ventricular_Ejection_Fraction",
                                      "MOTHER": "Use_Other_Noncardiac_Medication",
                                      "NSYANG": "Canadian_Cardiovascular_Society_Classification",
                                      "ORAGEN": "Use_Oral_Agents",
                                      "OTDIUR": "Use_Other_Diuretics", 
                                      'WT_KG': "Weight_KG", 
                                      'HT_CM': "Height_CM",
                                      'POSPDI': "Use_Potassium_Sparing_Diuretics", 
                                      "QUALAB": "Left_Ventricular_Function_Qualitative_Abnormal",
                                      "QUANEF": "Quantitative_Ejection_Fraction",
                                      'SKINRA': "Skin_Rash", 
                                      'SYNCOP': "Syncope"})
  
  features["Sex"].replace({1.0: "Female", 2.0: "Male"}, inplace=True)
  features['Baseline_Smoke_status'].replace({1.0: "Current", 2.0: "Ever", 3.0: "Never"}, inplace=True)
  features['Left_Ventricular_Evaluation_Documented_By'].replace({1.0: "Contrast_Ventriculography", 
                                                                 2.0: "Radionuclide_Ventriculography", 
                                                                 3.0: "Echocardiogram"}, inplace=True)
  features['Canadian_Cardiovascular_Society_Classification'].replace({1.0: "No symptoms of angina", 2.0: "I", 3.0: "II",
                                                                      4.0: "III+", 5.0: "III+"}, inplace=True)

  cols = ['History_Dizziness', 
          'History_Syncope', 
          'Use_Aspirin/Antiplatelet_Therapy', 
          'Use_Anti_Arrhythmic', 
          'Use_Anticoagulants', 
          'Beta_Blocker', 
          'Calcium_Blocker', 
          'Use_Other_Cardiac_Medication',
          'Cough',
          'Use_Other_Cardiac_Medication', 
          'Known_Diabetic',
          'History_Diabetes', 
          'Dizziness',
          'Headache',
          'Use_Diet_Control', 
          'Use_Digitalis', 
          'History_Angina', 
          'Angiographic_Coronary_Disease',
          'History_Coronary_Artery_Bypass_Graft', 
          'History_Intermittent_Claudication', 
          'History_Hypertension', 
          'History_Myocardial_Infarction', 
          'History_Percutaneous_Transluminal_Coronary_Angioplasty', 
          'History_Stroke', 
          'History_Transient_Ischemic_Attack', 
          'Use_Hormone_Replacement_Therapy', 
          'Use_Insulin', 
          'Use_Other_Noncardiac_Medication',
          'Use_Lipid_Lowering_Therapy', 
          'Use_Potassium_Sparing_Diuretics', 
          'Skin_Rash', 
          'Syncope',
          'Use_Oral_Agents',
          'Use_Other_Diuretics',
          'Use_Potassium_Sparing_Diuretics',
          'Left_Ventricular_Function_Qualitative_Abnormal']
  for col in cols:
    features[col].replace({1.0: "Yes", 2.0: "No"}, inplace=True)
  
  return outcomes, features